<table>
    <tr>
        <td><img src="https://raw.githubusercontent.com/Fabian830348/Bases_Datos/refs/heads/master/logo_ECI.png" width="300"/></td>
        <td>&nbsp;</td>
        <td>
            <h1 style="font-size:200%;color:blue;text-align:center"><FONT COLOR="blue">Entregable: K-means y PCA</FONT></h1>
        </td>
        <td>
            <p style="font-size:99%;text-align:center">Aprendizaje Automático de Máquina</p>
            <p style="font-size:115%;text-align:center">Pregrado MACC 2026-1</p>
            <p style="font-size:115%;text-align:center">Prof. Fabián Sánchez</p>
        </td>
    </tr>
</table>

# <FONT SIZE=5 COLOR="purple"> Instalación de librerías adicionales </FONT>

In [1]:
!pip install yellowbrick -q
!pip install pca -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.4/190.4 kB 4.5 MB/s eta 0:00:00


# <FONT SIZE=5 COLOR="purple"> Importación de librerías </FONT>

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples, pairwise_distances_argmin_min
from sklearn.neighbors import NearestNeighbors

from yellowbrick.cluster import KElbowVisualizer

import warnings
warnings.filterwarnings("ignore")

---
# <FONT COLOR="Purple"> 1. Clustering K-means — Segmentación de Clientes </FONT>

El conjunto de datos contiene información sobre:
- **CustomerID**: Identificación del cliente
- **Gender**: Género (Male/Female)
- **Age**: Edad
- **Annual Income**: Ingresos anuales en miles de $
- **Spending Score**: Puntos de Gastos (1-100)

El objetivo es realizar **segmentación de clientes** usando K-means.

## <FONT SIZE=4 COLOR="green"> 1.1 Carga de datos </FONT>

In [3]:
url = "https://raw.githubusercontent.com/Fabian830348/Bases_Datos/refs/heads/master/customers_clustering.csv"
df_raw = pd.read_csv(url)
print("Dimensiones:", df_raw.shape)
df_raw.head()

Dimensiones: (200, 5)


,CustomerID,Gender,Age,Annual Income (k$),Spending Score (1-100)
0,1,Male,19,15,39
1,2,Male,21,15,81
2,3,Female,20,16,6
3,4,Female,23,16,77
4,5,Female,31,17,40


In [4]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   CustomerID              200 non-null    int64 
 1   Gender                  200 non-null    object
 2   Age                     200 non-null    int64 
 3   Annual Income (k$)      200 non-null    int64 
 4   Spending Score (1-100)  200 non-null    int64 
dtypes: int64(4), object(1)
memory usage: 7.9+ KB


In [5]:
df_raw.describe()

,CustomerID,Age,Annual Income (k$),Spending Score (1-100)
count,200.000000,200.000000,200.000000,200.000000
mean,100.500000,38.850000,60.560000,50.200000
std,57.879185,13.969007,26.264721,25.823522
min,1.000000,18.000000,15.000000,1.000000
25%,50.750000,28.750000,41.500000,34.750000
50%,100.500000,36.000000,61.500000,50.000000
75%,150.250000,49.000000,78.000000,73.000000
max,200.000000,70.000000,137.000000,99.000000


## <FONT SIZE=4 COLOR="green"> 1.2 Exploración de variables </FONT>

In [6]:
# Frecuencia de la variable Gender
print("Distribución de Gender:")
print(df_raw['Genre'].value_counts())

fig = px.pie(df_raw, names='Genre', title='Distribución por Género')
fig.show()

Distribución de Gender:


KeyError: 'Genre'

In [ ]:
# Histogramas de variables numéricas
variables_num = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, var in zip(axes, variables_num):
    ax.hist(df_raw[var], bins=20, color='steelblue', edgecolor='white')
    ax.set_title(f'Histograma: {var}')
    ax.set_xlabel(var)
    ax.set_ylabel('Frecuencia')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots de variables numéricas con Plotly
df_long = df_raw[variables_num].melt(var_name='Variable', value_name='Valor')
fig = px.box(df_long, x='Variable', y='Valor', color='Variable',
             title='Boxplots de Variables Numéricas')
fig.update_layout(xaxis_title='Variables', yaxis_title='Valores')
fig.show()

In [ ]:
# Gráfico de dispersión entre Ingresos y Spending Score
fig = px.scatter(df_raw,
                 x='Annual Income (k$)',
                 y='Spending Score (1-100)',
                 color='Genre',
                 title='Ingresos vs Spending Score por Género')
fig.show()

## <FONT SIZE=4 COLOR="green"> 1.3 Preparación: sin variable Gender </FONT>

In [ ]:
# Seleccionamos solo variables numéricas (sin CustomerID ni Genre)
df_num = df_raw[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].copy()

# Escalamos los datos (importante para k-means porque usa distancias)
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_num),
                          columns=df_num.columns)
print("Datos escalados (primeras 5 filas):")
df_scaled.head()

## <FONT SIZE=4 COLOR="green"> 1.4 Aplicación de K-means con k inicial </FONT>

Probamos con **k=3** como valor inicial para explorar los clusters.

In [ ]:
# K-means con k=3
k_inicial = 3
kmeans_3 = KMeans(n_clusters=k_inicial, random_state=42)
kmeans_3.fit(df_scaled)

print(f"Inercia (WCSS) con k={k_inicial}: {kmeans_3.inertia_:.2f}")
print("\nCentroides:")
centroides_3 = pd.DataFrame(kmeans_3.cluster_centers_, columns=df_scaled.columns)
print(centroides_3)

In [ ]:
# Añadimos la etiqueta de cluster al DataFrame original
df_cluster3 = df_num.copy()
df_cluster3['cluster'] = kmeans_3.labels_.astype(str)

# Gráfico 3D
fig = px.scatter_3d(df_cluster3,
                    x='Age',
                    y='Annual Income (k$)',
                    z='Spending Score (1-100)',
                    color='cluster',
                    title=f'K-means con k={k_inicial} — Vista 3D')
fig.show()

In [ ]:
# Boxplots para interpretar clusters con k=3
fig_sub = make_subplots(rows=1, cols=3,
                        subplot_titles=['Age', 'Annual Income (k$)', 'Spending Score (1-100)'])

for i, var in enumerate(variables_num):
    for cluster_id in df_cluster3['cluster'].unique():
        vals = df_cluster3[df_cluster3['cluster'] == cluster_id][var]
        fig_sub.add_trace(
            go.Box(y=vals, name=f'Cluster {cluster_id}', showlegend=(i == 0)),
            row=1, col=i+1
        )

fig_sub.update_layout(title_text=f'Boxplots por Cluster (k={k_inicial})', height=500)
fig_sub.show()

## <FONT SIZE=4 COLOR="green"> 1.5 Selección del k óptimo: Método del Codo y Silueta </FONT>

In [ ]:
# --- Método del Codo ---
model_elbow = KMeans(random_state=42)
visualizer = KElbowVisualizer(model_elbow, k=(1, 12), timings=False)
visualizer.fit(df_scaled)
visualizer.show(show=False)
plt.title('Método del Codo')
plt.show()

In [ ]:
# Alternativa con Plotly para el codo
within = []
for k in range(1, 12):
    km = KMeans(n_clusters=k, random_state=42).fit(df_scaled)
    within.append(km.inertia_)

fig = px.line(x=list(range(1, 12)), y=within,
              labels={'x': 'Número de clusters k', 'y': 'Inercia (WCSS)'},
              title='Método del Codo — Inercia vs k',
              markers=True)
fig.show()

In [ ]:
# --- Método de la Silueta ---
models_sil = [KMeans(n_clusters=n, random_state=42).fit(df_scaled) for n in range(2, 12)]
silhouette_scores = [silhouette_score(df_scaled, m.labels_) for m in models_sil]

plt.figure(figsize=(8, 4))
plt.plot(list(range(2, 12)), silhouette_scores, 'bo-')
plt.xlabel('Número de clusters k', fontsize=13)
plt.ylabel('Silhouette Score', fontsize=13)
plt.title('Método de la Silueta')
plt.xticks(range(2, 12))
plt.grid(True, alpha=0.3)
plt.show()

k_optimo_sil = np.argmax(silhouette_scores) + 2
print(f"K óptimo según silueta: {k_optimo_sil} (score: {max(silhouette_scores):.4f})")

> **Observación:** El método del codo y la silueta pueden sugerir valores de k diferentes. El codo busca el punto donde la inercia deja de disminuir significativamente; la silueta busca el pico máximo de cohesión interna.

## <FONT SIZE=4 COLOR="green"> 1.6 K-means con k óptimo </FONT>

In [ ]:
# Usamos k=5 como valor óptimo (típico para este dataset de clientes)
# Ajusta según lo que indiquen tus gráficas del codo y silueta
k_opt = 5
kmeans_opt = KMeans(n_clusters=k_opt, random_state=42, init='k-means++')
kmeans_opt.fit(df_scaled)

print(f"K óptimo usado: {k_opt}")
print(f"Inercia: {kmeans_opt.inertia_:.2f}")
print(f"Silhouette Score: {silhouette_score(df_scaled, kmeans_opt.labels_):.4f}")

In [ ]:
# DataFrame con etiquetas de cluster óptimo
df_cluster_opt = df_num.copy()
df_cluster_opt['cluster'] = kmeans_opt.labels_.astype(str)

# Media por cluster
print("Media de cada variable por cluster:")
df_cluster_opt.groupby('cluster').mean().round(2)

In [ ]:
# Gráfico 3D con k óptimo
fig = px.scatter_3d(df_cluster_opt,
                    x='Age',
                    y='Annual Income (k$)',
                    z='Spending Score (1-100)',
                    color='cluster',
                    title=f'K-means con k={k_opt} — Vista 3D')
fig.show()

In [ ]:
# Boxplots por variable para k óptimo
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
palette = {str(i): c for i, c in enumerate(['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6'])}

for ax, var in zip(axes, variables_num):
    sns.boxplot(data=df_cluster_opt, x='cluster', y=var, palette=palette, ax=ax)
    ax.set_title(var)

plt.suptitle(f'Boxplots por Cluster (k={k_opt})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## <FONT SIZE=4 COLOR="green"> 1.7 Representante de cada cluster y vecinos más cercanos </FONT>

In [ ]:
# Representante: registro más cercano al centroide
closest_indices, _ = pairwise_distances_argmin_min(kmeans_opt.cluster_centers_, df_scaled)

print("=== Representante de cada cluster (registro más cercano al centroide) ===")
for cluster_id, idx in enumerate(closest_indices):
    print(f"\nCluster {cluster_id}: CustomerID = {df_raw.loc[idx, 'CustomerID']}")
    print(df_num.iloc[idx])

In [ ]:
# Vecinos más cercanos al representante de cada cluster
nn_model = NearestNeighbors(n_neighbors=4)  # 1 propio + 3 vecinos
nn_model.fit(df_scaled)

print("=== 3 Vecinos más cercanos al representante de cada cluster ===")
for cluster_id, idx in enumerate(closest_indices):
    punto = df_scaled.iloc[[idx]]
    distancias, indices = nn_model.kneighbors(punto)
    vecinos_idx = indices[0][1:]  # excluimos el propio punto
    print(f"\nCluster {cluster_id} (representante: CustomerID {df_raw.loc[idx, 'CustomerID']}):")
    print(f"Vecinos más cercanos (CustomerID): {df_raw.loc[vecinos_idx, 'CustomerID'].values}")
    print(df_num.iloc[vecinos_idx])

## <FONT SIZE=4 COLOR="green"> 1.8 K-means incluyendo la variable Gender </FONT>

In [ ]:
# Codificamos la variable Genre (Male=0, Female=1)
le = LabelEncoder()
df_con_genre = df_num.copy()
df_con_genre['Genre'] = le.fit_transform(df_raw['Genre'])

# Escalamos incluyendo Genre
scaler2 = StandardScaler()
df_scaled_genre = pd.DataFrame(scaler2.fit_transform(df_con_genre),
                                 columns=df_con_genre.columns)

# K-means con el mismo k óptimo
kmeans_genre = KMeans(n_clusters=k_opt, random_state=42, init='k-means++')
kmeans_genre.fit(df_scaled_genre)

print(f"Inercia (con Genre): {kmeans_genre.inertia_:.2f}")
print(f"Silhouette Score (con Genre): {silhouette_score(df_scaled_genre, kmeans_genre.labels_):.4f}")

In [ ]:
df_cluster_genre = df_num.copy()
df_cluster_genre['Genre'] = df_raw['Genre']
df_cluster_genre['cluster'] = kmeans_genre.labels_.astype(str)

# Comparación de distribución de género por cluster
fig = px.histogram(df_cluster_genre, x='cluster', color='Genre', barmode='group',
                   title=f'Distribución de Género por Cluster (k={k_opt})')
fig.show()

print("\nDistribución de Genre por cluster:")
print(pd.crosstab(df_cluster_genre['cluster'], df_cluster_genre['Genre']))

In [ ]:
# Gráfico 3D con Genre
fig = px.scatter_3d(df_cluster_genre,
                    x='Age',
                    y='Annual Income (k$)',
                    z='Spending Score (1-100)',
                    color='cluster',
                    symbol='Genre',
                    title=f'K-means con Genre (k={k_opt})')
fig.show()

## <FONT SIZE=4 COLOR="green"> 1.9 Análisis y Conclusiones — K-means </FONT>

**Comparación de los clusters obtenidos sin y con la variable Genre:**

In [ ]:
print("=== Media por cluster SIN Genre ===")
print(df_cluster_opt.groupby('cluster').mean().round(2))
print()
print("=== Media por cluster CON Genre ===")
medias_genre = df_cluster_genre.copy()
medias_genre['Genre_num'] = le.transform(df_cluster_genre['Genre'])
print(medias_genre.groupby('cluster')[['Age','Annual Income (k$)','Spending Score (1-100)','Genre_num']].mean().round(2))

> **Conclusiones K-means:**
>
> - Se identificaron grupos diferenciados según los ingresos anuales y el puntaje de gasto.
> - El método del codo y la silueta pueden sugerir valores distintos de k; es necesario evaluar ambos y elegir el que tenga más sentido desde el dominio del problema.
> - Incluir la variable Genre no cambia sustancialmente la segmentación dado que el género no es el principal discriminador entre los grupos de consumidores en este dataset.
> - Los representantes de cada cluster son los clientes más "típicos" de su segmento, útiles para campañas de marketing personalizadas.

---
# <FONT COLOR="Purple"> 2. Análisis de Componentes Principales (PCA) — Delitos en Colombia </FONT>

Se analizan datos de delitos en Colombia para el año 2017. El objetivo es:
1. Cargar y explorar los datos
2. Aplicar PCA e identificar las componentes
3. Construir el Biplot
4. Concluir

## <FONT SIZE=4 COLOR="green"> 2.1 Carga y Exploración </FONT>

In [ ]:
url_delitos = "https://raw.githubusercontent.com/Fabian830348/Bases_Datos/refs/heads/master/delitos_colombia.csv"
delitos = pd.read_csv(url_delitos, sep=";", decimal=",", index_col=0)
print("Dimensiones:", delitos.shape)
delitos.head(10)

In [ ]:
delitos.info()

In [ ]:
delitos.describe().round(2)

In [ ]:
# Verificamos datos faltantes
print("Datos faltantes por variable:")
print(delitos.isnull().sum())

In [ ]:
# Boxplots de todas las variables
df_long_delitos = delitos.melt(var_name='Delito', value_name='Casos')
fig = px.box(df_long_delitos, x='Delito', y='Casos', color='Delito',
             title='Distribución de Delitos por Departamento (Colombia 2017)')
fig.update_layout(xaxis_tickangle=-45, showlegend=False)
fig.show()

In [ ]:
# Matriz de correlación
corr_matrix = delitos.corr()

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr_matrix,
            annot=True,
            fmt='.2f',
            cmap='coolwarm',
            center=0,
            ax=ax)
ax.set_title('Matriz de Correlación — Delitos Colombia 2017', fontsize=14)
plt.tight_layout()
plt.show()

> **Observación:** Cuando hay correlaciones altas entre variables, PCA es una técnica adecuada para reducir la dimensionalidad sin perder mucha información.

## <FONT SIZE=4 COLOR="green"> 2.2 Análisis PCA </FONT>

In [ ]:
# Estandarización de los datos (requisito para PCA)
scaler_pca = StandardScaler()
delitos_std = scaler_pca.fit_transform(delitos)
delitos_std_df = pd.DataFrame(delitos_std, columns=delitos.columns, index=delitos.index)

print("Media después de escalar (deben ser ~0):", delitos_std_df.mean().round(4).values)
print("Std después de escalar (deben ser ~1):", delitos_std_df.std().round(4).values)

In [ ]:
# Aplicamos PCA con todas las componentes
pca_full = PCA()
pca_full.fit(delitos_std)

# Varianza explicada por cada componente
var_ratio = pca_full.explained_variance_ratio_
var_acum  = np.cumsum(var_ratio)

print("Varianza explicada por componente:")
for i, (vr, va) in enumerate(zip(var_ratio, var_acum)):
    print(f"  PC{i+1}: {vr*100:.2f}%   Acumulada: {va*100:.2f}%")

In [ ]:
# Gráfico de varianza explicada (Scree Plot)
n_comp = len(var_ratio)
x_labels = [f'PC{i+1}' for i in range(n_comp)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Varianza individual
axes[0].bar(x_labels, var_ratio * 100, color='steelblue', edgecolor='white')
axes[0].set_title('Varianza Explicada por Componente')
axes[0].set_ylabel('% Varianza Explicada')
axes[0].set_xlabel('Componente Principal')
axes[0].tick_params(axis='x', rotation=45)

# Varianza acumulada
axes[1].plot(x_labels, var_acum * 100, 'o-', color='darkorange', linewidth=2)
axes[1].axhline(y=80, color='red', linestyle='--', alpha=0.7, label='80%')
axes[1].axhline(y=90, color='green', linestyle='--', alpha=0.7, label='90%')
axes[1].set_title('Varianza Acumulada Explicada')
axes[1].set_ylabel('% Varianza Acumulada')
axes[1].set_xlabel('Componente Principal')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Seleccionamos las componentes que expliquen al menos el 80% de la varianza
n_componentes = np.argmax(var_acum >= 0.80) + 1
print(f"Número de componentes para explicar >= 80% de varianza: {n_componentes}")
print(f"Varianza acumulada con {n_componentes} componentes: {var_acum[n_componentes-1]*100:.2f}%")

In [ ]:
# Aplicamos PCA con las componentes seleccionadas
pca = PCA(n_components=n_componentes)
componentes = pca.fit_transform(delitos_std)

col_names = [f'PC{i+1}' for i in range(n_componentes)]
df_pca = pd.DataFrame(componentes, columns=col_names, index=delitos.index)
print("Primeras filas de las componentes principales:")
df_pca.head(10)

In [ ]:
# Loadings (pesos de cada variable original en cada componente)
loadings = pd.DataFrame(
    pca.components_.T,
    columns=col_names,
    index=delitos.columns
)
print("Loadings (contribución de cada variable a cada componente):")
loadings.round(3)

In [ ]:
# Heatmap de los loadings
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(loadings,
            annot=True,
            fmt='.3f',
            cmap='RdBu_r',
            center=0,
            ax=ax)
ax.set_title('Loadings de Componentes Principales', fontsize=14)
plt.tight_layout()
plt.show()

## <FONT SIZE=4 COLOR="green"> 2.3 Biplot </FONT>

In [ ]:
def biplot(scores, loadings, labels_obs=None, labels_var=None, escala=1.5):
    """
    Construye un Biplot de PC1 vs PC2.
    - scores: coordenadas de las observaciones en el espacio PCA
    - loadings: contribución de cada variable original
    - labels_obs: etiquetas para las observaciones (departamentos)
    - labels_var: nombres de las variables originales
    - escala: factor de escalado para las flechas de los loadings
    """
    fig, ax = plt.subplots(figsize=(12, 9))

    # Puntos (observaciones)
    ax.scatter(scores[:, 0], scores[:, 1], alpha=0.6, color='steelblue', s=60, zorder=3)

    if labels_obs is not None:
        for i, lbl in enumerate(labels_obs):
            ax.annotate(lbl, (scores[i, 0], scores[i, 1]),
                        fontsize=7, ha='right', alpha=0.8)

    # Flechas (variables)
    max_score = np.max(np.abs(scores[:, :2]))
    factor = max_score * escala

    for j, var in enumerate(labels_var):
        lx, ly = loadings[j, 0] * factor, loadings[j, 1] * factor
        ax.annotate('', xy=(lx, ly), xytext=(0, 0),
                    arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
        ax.text(lx * 1.05, ly * 1.05, var, color='red', fontsize=9, ha='center')

    ax.axhline(0, color='gray', lw=0.8, linestyle='--')
    ax.axvline(0, color='gray', lw=0.8, linestyle='--')
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=13)
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=13)
    ax.set_title('Biplot — Delitos Colombia 2017', fontsize=14)
    plt.tight_layout()
    plt.show()


biplot(
    scores=componentes,
    loadings=pca.components_.T,
    labels_obs=delitos.index.tolist(),
    labels_var=delitos.columns.tolist()
)

In [ ]:
# Biplot interactivo con Plotly
df_biplot = df_pca.copy().reset_index()
df_biplot.columns = ['Departamento'] + col_names

fig_biplot = px.scatter(df_biplot, x='PC1', y='PC2',
                        text='Departamento',
                        title='Biplot Interactivo — Delitos Colombia 2017',
                        labels={'PC1': f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)',
                                'PC2': f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)'})
fig_biplot.update_traces(textposition='top center', marker=dict(size=8))

# Agregar flechas de los loadings
max_score = max(abs(componentes[:,0]).max(), abs(componentes[:,1]).max())
factor_bi = max_score * 1.2

for j, var in enumerate(delitos.columns):
    lx = pca.components_[0, j] * factor_bi
    ly = pca.components_[1, j] * factor_bi
    fig_biplot.add_annotation(
        x=lx, y=ly, ax=0, ay=0,
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True, arrowhead=2, arrowcolor='red', arrowwidth=1.5
    )
    fig_biplot.add_annotation(
        x=lx*1.08, y=ly*1.08,
        text=f'<b>{var}</b>', showarrow=False,
        font=dict(color='red', size=10)
    )

fig_biplot.add_hline(y=0, line_dash='dash', line_color='gray', line_width=0.8)
fig_biplot.add_vline(x=0, line_dash='dash', line_color='gray', line_width=0.8)
fig_biplot.show()

## <FONT SIZE=4 COLOR="green"> 2.4 Conclusiones — PCA </FONT>

In [ ]:
# Resumen de resultados PCA
resumen = pd.DataFrame({
    'Componente': col_names,
    'Varianza Explicada (%)': (pca.explained_variance_ratio_ * 100).round(2),
    'Varianza Acumulada (%)': (np.cumsum(pca.explained_variance_ratio_) * 100).round(2)
})
print(resumen.to_string(index=False))

In [ ]:
# Ranking de variables más influyentes en PC1 y PC2
print("\n=== Variables más influyentes en PC1 (ordenadas por |loading|) ===")
print(loadings['PC1'].abs().sort_values(ascending=False).round(4))

print("\n=== Variables más influyentes en PC2 (ordenadas por |loading|) ===")
print(loadings['PC2'].abs().sort_values(ascending=False).round(4))

> **Conclusiones PCA:**
>
> - Las primeras componentes principales permiten resumir la variabilidad de todos los tipos de delitos registrados en Colombia en 2017.
> - PC1 captura principalmente la magnitud general de la criminalidad en los departamentos (a mayor puntaje en PC1, mayor actividad delictiva global).
> - PC2 diferencia departamentos según el tipo específico de delito predominante.
> - El Biplot permite identificar qué departamentos tienen perfiles delictivos similares y cuáles son atípicos.
> - Variables con flechas en la misma dirección están positivamente correlacionadas; flechas opuestas indican correlación negativa.
> - La longitud de la flecha indica el nivel de contribución de cada variable a las componentes principales mostradas.